In [1]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

URL = "https://www.manipalcigna.com/diet/fever-diet"
OUTPUT_FILE = "fever_diet_food_items.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

In [2]:
def clean_text(value):
    """Collapse whitespace while preserving spaces between words."""
    return re.sub(r"\s+", " ", str(value or "")).strip()


def get_cell_text(cell):
    """
    Read the complete cell as continuous text.
    HTML line breaks are converted to spaces, not treated as food separators.
    """
    return clean_text(cell.get_text(" ", strip=True))


def split_food_items(text):
    """
    Split only on explicit list punctuation.
    Never split on spaces or newline characters.
    """
    text = clean_text(text)
    return [
        clean_text(item)
        for item in re.split(r"\s*[,;•●▪]\s*", text)
        if clean_text(item)
    ]

In [3]:
response = requests.get(URL, headers=HEADERS, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print("Page downloaded successfully.")
print("Tables found:", len(soup.find_all("table")))

Page downloaded successfully.
Tables found: 2


In [4]:
food_items = []

for table in soup.find_all("table"):
    rows = table.find_all("tr")
    if not rows:
        continue

    header_cells = rows[0].find_all(["th", "td"])
    headers = [get_cell_text(cell).lower() for cell in header_cells]

    if not any("food group" in header for header in headers):
        continue

    for row in rows[1:]:
        cells = row.find_all(["td", "th"])

        if len(cells) < 2:
            continue

        foods_cell = get_cell_text(cells[1])

        for item in split_food_items(foods_cell):
            if item:
                food_items.append(item)

    break

print("Food items extracted:", len(food_items))

Food items extracted: 21


In [5]:
seen = set()
unique_food_items = []

for item in food_items:
    key = item.casefold()
    if key not in seen:
        seen.add(key)
        unique_food_items.append(item)

df = pd.DataFrame({"food_item": unique_food_items})

print("Unique food items:", len(df))
display(df)

Unique food items: 21


,food_item
0,Water
1,ORS
2,coconut water
3,warm milk
4,Khichdi
5,plain rice
6,oats
7,Moong dal
8,masoor dal
9,Curd


In [6]:
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"Saved {len(df)} food items to: {OUTPUT_FILE}")
print("CSV columns:", df.columns.tolist())

Saved 21 food items to: fever_diet_food_items.csv
CSV columns: ['food_item']
